In [1]:
import warnings
warnings.filterwarnings("ignore")

from yfinance import download
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2, norm

# Análise de Retornos - Dashboard Completo

Análise quantitativa de retornos e risco para um ativo específico.

**Indicadores incluídos:**
- Retornos diários/mensais com distribuição
- Volatilidade histórica (rolling)
- Métricas de risco: VaR, CVaR, Max Drawdown
- Relação Retorno vs Risco com regressão
- Análise de topos e fundos

In [2]:
# ══════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ══════════════════════════════════════════════════════════════
param = {
    'ticker': '^BVSP',
    'interval': '1d',
    'period': '10y',
    'column': 'Close',
    'vol_window': 21,        # Janela para volatilidade
    'sharpe_window': 63,     # Janela para rolling Sharpe (3 meses)
    'var_confidence': 0.95,  # Nível de confiança VaR
}

# Paleta de cores profissional
CORES = {
    'principal': '#1a1a2e',
    'tendencia': '#6c5ce7',
    'ma200': '#00cec9',
    'media_acc': '#fd79a8',
    'positivo': '#00b894',
    'negativo': '#d63031',
    'neutro': '#636e72',
    'fundo_hist': '#74b9ff',
    'drawdown': '#e17055',
    'volatilidade': '#a29bfe',
}

# ══════════════════════════════════════════════════════════════
# DOWNLOAD DOS DADOS
# ══════════════════════════════════════════════════════════════
df = download(
    param['ticker'], 
    period=param['period'], 
    interval=param['interval'], 
    progress=False, 
    auto_adjust=False
).droplevel(1, axis=1)

print(f"Dados carregados: {len(df)} registros de {df.index[0].strftime('%d/%m/%Y')} a {df.index[-1].strftime('%d/%m/%Y')}")

# ══════════════════════════════════════════════════════════════
# CÁLCULO DOS INDICADORES
# ══════════════════════════════════════════════════════════════
col = param['column']

# Retornos
df['ret'] = df[col].pct_change()
df['ret_log'] = np.log(df[col] / df[col].shift(1))

# Retorno acumulado
df['ret_acum'] = (1 + df['ret']).cumprod() - 1

# Volatilidade rolling
df['vol'] = df['ret'].rolling(window=param['vol_window']).std() * np.sqrt(252)

# Médias móveis de retorno
df['ret_max_21'] = df['ret'].rolling(param['vol_window']).max()
df['ret_mean_21'] = df['ret'].rolling(param['vol_window']).mean()
df['ret_min_21'] = df['ret'].rolling(param['vol_window']).min()

# Rolling Sharpe (assumindo rf=0)
df['rolling_sharpe'] = (
    df['ret'].rolling(param['sharpe_window']).mean() * 252
) / (df['ret'].rolling(param['sharpe_window']).std() * np.sqrt(252))

# Drawdown
preco_maximo = df[col].expanding().max()
df['drawdown'] = (df[col] - preco_maximo) / preco_maximo

# Topos e fundos (função auxiliar)
def encontrar_topos_fundos(series):
    topos, fundos = [], []
    for i in range(1, len(series)-1):
        if series[i] < series[i-1] and series[i-1] > series[i-2]:
            topos.append(i-1)
        elif series[i] > series[i-1] and series[i-1] < series[i-2]:
            fundos.append(i-1)
    return topos, fundos

topos, fundos = encontrar_topos_fundos(df[col].values)

df.dropna(inplace=True)

Dados carregados: 2485 registros de 23/02/2016 a 23/02/2026


In [3]:
# ══════════════════════════════════════════════════════════════
# MÉTRICAS DE RISCO E RETORNO
# ══════════════════════════════════════════════════════════════
retornos = df['ret'].dropna()
col = param['column']

# Retorno total e CAGR
dias = len(df)
anos = dias / 252
retorno_total = (df[col].iloc[-1] / df[col].iloc[0] - 1) * 100
cagr = ((df[col].iloc[-1] / df[col].iloc[0]) ** (1/anos) - 1) * 100

# Volatilidade anualizada
volatilidade = retornos.std() * np.sqrt(252) * 100

# Max Drawdown
max_drawdown = df['drawdown'].min() * 100

# Sharpe Ratio (rf=0)
sharpe = (retornos.mean() * 252) / (retornos.std() * np.sqrt(252))

# VaR Paramétrico (95%)
var_95 = norm.ppf(1 - param['var_confidence']) * retornos.std() * 100
var_99 = norm.ppf(0.01) * retornos.std() * 100

# VaR Histórico
var_hist_95 = retornos.quantile(0.05) * 100
var_hist_99 = retornos.quantile(0.01) * 100

# CVaR (Expected Shortfall)
cvar_95 = retornos[retornos <= retornos.quantile(0.05)].mean() * 100
cvar_99 = retornos[retornos <= retornos.quantile(0.01)].mean() * 100

# Estatísticas descritivas
skewness = retornos.skew()
kurtosis = retornos.kurtosis()

# Valores atuais
ret_atual = retornos.iloc[-1] * 100
vol_atual = df['vol'].iloc[-1] * 100
sharpe_atual = df['rolling_sharpe'].iloc[-1]
dd_atual = df['drawdown'].iloc[-1] * 100

# Percentis
percentil_ret = (retornos < retornos.iloc[-1]).sum() / len(retornos) * 100
percentil_vol = (df['vol'].dropna() < df['vol'].iloc[-1]).sum() / len(df['vol'].dropna()) * 100

metricas = {
    'Preço Atual': f"R$ {df[col].iloc[-1]:.2f}",
    'Retorno Total': f"{retorno_total:.1f}%",
    'CAGR': f"{cagr:.1f}% a.a.",
    'Volatilidade': f"{volatilidade:.1f}% a.a.",
    'Max Drawdown': f"{max_drawdown:.1f}%",
    'Sharpe Ratio': f"{sharpe:.2f}",
    'VaR 95% (Param)': f"{var_95:.2f}%",
    'VaR 99% (Param)': f"{var_99:.2f}%",
    'CVaR 95%': f"{cvar_95:.2f}%",
    'CVaR 99%': f"{cvar_99:.2f}%",
    'Skewness': f"{skewness:.2f}",
    'Kurtosis': f"{kurtosis:.2f}",
    'Ret. Hoje': f"{ret_atual:.2f}%",
    'Vol. Atual': f"{vol_atual:.1f}%",
    'Percentil Ret': f"{percentil_ret:.0f}%",
}

print(f"{'═'*55}")
print(f"  RESUMO DE RISCO - {param['ticker'].upper()}")
print(f"{'═'*55}")
for k, v in metricas.items():
    print(f"  {k:.<22} {v:>18}")
print(f"{'═'*55}")

═══════════════════════════════════════════════════════
  RESUMO DE RISCO - ^BVSP
═══════════════════════════════════════════════════════
  Preço Atual...........       R$ 189787.48
  Retorno Total.........             284.6%
  CAGR..................         15.0% a.a.
  Volatilidade..........         22.8% a.a.
  Max Drawdown..........             -46.8%
  Sharpe Ratio..........               0.73
  VaR 95% (Param).......             -2.37%
  VaR 99% (Param).......             -3.35%
  CVaR 95%..............             -3.28%
  CVaR 99%..............             -6.04%
  Skewness..............              -0.93
  Kurtosis..............              17.33
  Ret. Hoje.............             -0.39%
  Vol. Atual............              19.6%
  Percentil Ret.........                32%
═══════════════════════════════════════════════════════


In [4]:
# ══════════════════════════════════════════════════════════════
# DASHBOARD DE RETORNO E VOLATILIDADE
# ══════════════════════════════════════════════════════════════

fig = make_subplots(
    rows=3,
    cols=2,
    shared_xaxes='columns',
    vertical_spacing=0.06,
    horizontal_spacing=0.04,
    column_widths=[0.70, 0.30],
    row_heights=[0.35, 0.35, 0.30],
    specs=[
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None],
    ],
    subplot_titles=(
        'Retorno Diário', 'Distribuição',
        'Volatilidade Anualizada (21d)', 'Distribuição',
        'Drawdown'
    )
)

# ══════════════════════════════════════════════════════════════
# ROW 1 — RETORNO
# ══════════════════════════════════════════════════════════════

# Retorno - Linha
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['ret'],
    name='Retorno',
    line=dict(color=CORES['principal'], width=1),
    hovertemplate='%{y:.2%}<extra></extra>'
), row=1, col=1)

# Bandas de retorno
fig.add_trace(go.Scatter(
    x=df.index, y=df['ret_max_21'],
    name='Max 21d', line=dict(color=CORES['positivo'], width=0.8, dash='dot'),
    opacity=0.6, showlegend=False
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['ret_min_21'],
    name='Min 21d', line=dict(color=CORES['negativo'], width=0.8, dash='dot'),
    opacity=0.6, showlegend=False
), row=1, col=1)

fig.add_hline(y=0, row=1, col=1, line_color=CORES['neutro'], line_width=1)

# VaR lines
fig.add_hline(y=var_hist_95/100, row=1, col=1, line_color=CORES['negativo'], 
              line_dash='dash', line_width=1, opacity=0.7,
              annotation_text='VaR 95%', annotation_position='bottom left')

# Retorno - Histograma
ret_vals = df['ret'].dropna()
fig.add_trace(go.Histogram(
    x=ret_vals,
    nbinsx=60,
    opacity=0.8,
    marker=dict(color=CORES['fundo_hist'], line=dict(color='white', width=0.3)),
    showlegend=False,
    hovertemplate='%{x:.2%}<br>Freq: %{y}<extra></extra>'
), row=1, col=2)

fig.add_vline(x=0, row=1, col=2, line_color=CORES['neutro'], line_width=1.5)
fig.add_vline(x=ret_vals.mean(), row=1, col=2, line_color=CORES['tendencia'], 
              line_width=2, line_dash='dash')
fig.add_vline(x=ret_vals.iloc[-1], row=1, col=2, line_color=CORES['principal'], line_width=2.5)

# VaR no histograma
fig.add_vline(x=var_hist_95/100, row=1, col=2, line_color=CORES['negativo'], 
              line_width=2, line_dash='dash')

fig.add_annotation(
    x=ret_vals.iloc[-1], y=1.02, yref='y2 domain',
    text=f"<b>{ret_vals.iloc[-1]:.2%}</b>",
    showarrow=False, font=dict(size=10, color=CORES['principal']),
    row=1, col=2
)

# ══════════════════════════════════════════════════════════════
# ROW 2 — VOLATILIDADE
# ══════════════════════════════════════════════════════════════

# Volatilidade - Linha
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['vol'],
    name='Volatilidade',
    line=dict(color=CORES['volatilidade'], width=1.2),
    hovertemplate='%{y:.1%}<extra></extra>'
), row=2, col=1)

# Quartis
q25_vol = df['vol'].quantile(0.25)
q50_vol = df['vol'].quantile(0.50)
q75_vol = df['vol'].quantile(0.75)

fig.add_hline(y=q25_vol, row=2, col=1, line_color=CORES['positivo'], 
              line_dash='dot', line_width=1, opacity=0.7)
fig.add_hline(y=q50_vol, row=2, col=1, line_color=CORES['tendencia'], 
              line_dash='dash', line_width=1.2,
              annotation_text='Mediana', annotation_position='top left')
fig.add_hline(y=q75_vol, row=2, col=1, line_color=CORES['negativo'], 
              line_dash='dot', line_width=1, opacity=0.7)

# Volatilidade - Histograma
vol_vals = df['vol'].dropna()
fig.add_trace(go.Histogram(
    x=vol_vals,
    nbinsx=50,
    opacity=0.8,
    marker=dict(color=CORES['volatilidade'], line=dict(color='white', width=0.3)),
    showlegend=False,
    hovertemplate='%{x:.1%}<br>Freq: %{y}<extra></extra>'
), row=2, col=2)

fig.add_vline(x=vol_vals.mean(), row=2, col=2, line_color=CORES['tendencia'], 
              line_width=2, line_dash='dash')
fig.add_vline(x=vol_vals.iloc[-1], row=2, col=2, line_color=CORES['principal'], line_width=2.5)

fig.add_annotation(
    x=vol_vals.iloc[-1], y=1.02, yref='y4 domain',
    text=f"<b>{vol_vals.iloc[-1]:.1%}</b>",
    showarrow=False, font=dict(size=10, color=CORES['principal']),
    row=2, col=2
)

# ══════════════════════════════════════════════════════════════
# ROW 3 — DRAWDOWN
# ══════════════════════════════════════════════════════════════

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['drawdown'],
    name='Drawdown',
    fill='tozeroy',
    line=dict(color=CORES['drawdown'], width=1),
    fillcolor='rgba(225, 112, 85, 0.4)',
    showlegend=False,
    hovertemplate='%{y:.1%}<extra></extra>'
), row=3, col=1)

fig.add_hline(
    y=df['drawdown'].min(),
    row=3, col=1,
    line_color=CORES['negativo'],
    line_dash='dash',
    line_width=1.5,
)

fig.add_annotation(
    x=df.index[0], y=df['drawdown'].min(),
    text=f"  Max DD: {df['drawdown'].min():.1%}",
    showarrow=False, xanchor='left',
    font=dict(size=10, color=CORES['negativo']),
    row=3, col=1
)

# ══════════════════════════════════════════════════════════════
# LAYOUT
# ══════════════════════════════════════════════════════════════

fig.update_layout(
    template='plotly_white',
    height=900,
    width=1400,
    margin=dict(l=60, r=60, t=80, b=40),
    hovermode='x unified',
    title=dict(
        text=f"<b>{param['ticker'].upper()}</b> — Análise de Retorno e Risco",
        font=dict(size=20, color=CORES['principal']),
        x=0.5, xanchor='center'
    ),
    legend=dict(
        orientation='h', y=1.02, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.9)',
        font=dict(size=10)
    ),
    font=dict(family='Segoe UI, Arial', size=10, color=CORES['principal'])
)

# Eixos Y
fig.update_yaxes(title_text='Retorno', title_font=dict(size=10), row=1, col=1)
fig.update_yaxes(title_text='Vol. Anual', title_font=dict(size=10), row=2, col=1)
fig.update_yaxes(title_text='Drawdown', title_font=dict(size=10), row=3, col=1)

fig.update_yaxes(showticklabels=False, row=1, col=2)
fig.update_yaxes(showticklabels=False, row=2, col=2)

# Formato percentual
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_yaxes(tickformat='.0%', row=3, col=1)

# Grid
fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')

# Datas
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(showticklabels=True, row=3, col=1, tickformat='%b/%Y')

fig.show()

In [5]:
# ══════════════════════════════════════════════════════════════
# SCATTER: RETORNO vs VOLATILIDADE + ELIPSE DE CONFIANÇA
# ══════════════════════════════════════════════════════════════

# Dados
x = df['vol'].dropna()
y = df.loc[x.index, 'ret']

x_atual = x.iloc[-1]
y_atual = y.iloc[-1]

# Regressão linear
coef = np.polyfit(x, y, 1)
poly = np.poly1d(coef)
x_reg = np.linspace(x.min(), x.max(), 200)
y_reg = poly(x_reg)

# Elipse de confiança 95%
cov = np.cov(x, y)
vals, vecs = np.linalg.eig(cov)
order = vals.argsort()[::-1]
vals, vecs = vals[order], vecs[:, order]

theta = np.linspace(0, 2 * np.pi, 300)
chi_val = np.sqrt(chi2.ppf(0.95, 2))

ellipse_x = chi_val * np.sqrt(vals[0]) * np.cos(theta)
ellipse_y = chi_val * np.sqrt(vals[1]) * np.sin(theta)
ellipse = np.dot(vecs, np.vstack([ellipse_x, ellipse_y]))
ellipse_x_rot = ellipse[0] + x.mean()
ellipse_y_rot = ellipse[1] + y.mean()

# Figura
fig = go.Figure()

# Dispersão
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    name='Observações',
    marker=dict(
        size=6,
        color=CORES['fundo_hist'],
        opacity=0.5,
        line=dict(color='white', width=0.5)
    ),
    hovertemplate='Vol: %{x:.2%}<br>Ret: %{y:.2%}<extra></extra>'
))

# Regressão
fig.add_trace(go.Scatter(
    x=x_reg, y=y_reg,
    mode='lines',
    name=f'Regressão (β={coef[0]:.2f})',
    line=dict(color=CORES['neutro'], dash='dash', width=2)
))

# Elipse 95%
fig.add_trace(go.Scatter(
    x=ellipse_x_rot, y=ellipse_y_rot,
    mode='lines',
    name='Elipse 95%',
    line=dict(color=CORES['negativo'], dash='dash', width=2)
))

# Ponto atual
fig.add_trace(go.Scatter(
    x=[x_atual], y=[y_atual],
    mode='markers',
    name='Atual',
    marker=dict(
        size=14,
        color=CORES['negativo'],
        symbol='diamond',
        line=dict(color='black', width=1.5)
    )
))

# Centro
fig.add_trace(go.Scatter(
    x=[x.mean()], y=[y.mean()],
    mode='markers',
    name='Média',
    marker=dict(
        size=12,
        color=CORES['tendencia'],
        symbol='x',
        line=dict(width=2)
    )
))

# Anotação do ponto atual
fig.add_annotation(
    x=x_atual, y=y_atual,
    text=f"<b>Vol: {x_atual:.1%}, Ret: {y_atual:.2%}</b>",
    showarrow=True, arrowhead=2,
    ax=40, ay=-40,
    font=dict(size=11, color=CORES['negativo']),
    bgcolor='white',
    bordercolor=CORES['negativo'],
    borderwidth=1
)

# Layout
fig.update_layout(
    template='plotly_white',
    height=650,
    width=1000,
    title=dict(
        text=f"<b>{param['ticker'].upper()}</b> — Relação Retorno vs Volatilidade",
        font=dict(size=18, color=CORES['principal']),
        x=0.5, xanchor='center'
    ),
    xaxis_title='Volatilidade Anualizada',
    yaxis_title='Retorno Diário',
    legend=dict(
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor=CORES['neutro'],
        borderwidth=0.5,
        font=dict(size=10)
    ),
    font=dict(family='Segoe UI, Arial', size=10, color=CORES['principal'])
)

fig.update_xaxes(
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    tickformat='.0%'
)
fig.update_yaxes(
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    tickformat='.1%'
)

fig.show()

In [6]:
# ══════════════════════════════════════════════════════════════
# PREÇO COM TOPOS/FUNDOS + ROLLING SHARPE
# ══════════════════════════════════════════════════════════════

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.60, 0.40],
    subplot_titles=('Preço com Topos e Fundos', 'Rolling Sharpe Ratio')
)

# ══════════════════════════════════════════════════════════════
# ROW 1 — PREÇO COM TOPOS/FUNDOS
# ══════════════════════════════════════════════════════════════

col = param['column']

# Linha de preço
fig.add_trace(go.Scatter(
    x=df.index,
    y=df[col],
    name='Preço',
    line=dict(color=CORES['principal'], width=2),
    hovertemplate='R$ %{y:.2f}<extra></extra>'
), row=1, col=1)

# Topos
if len(topos) > 0:
    topos_valid = [t for t in topos if t < len(df)]
    fig.add_trace(go.Scatter(
        x=df.index[topos_valid],
        y=df[col].values[topos_valid],
        mode='markers',
        name='Topos',
        marker=dict(
            size=10,
            color=CORES['negativo'],
            symbol='triangle-down',
            line=dict(color='white', width=1)
        ),
        hovertemplate='Topo: R$ %{y:.2f}<extra></extra>'
    ), row=1, col=1)

# Fundos
if len(fundos) > 0:
    fundos_valid = [f for f in fundos if f < len(df)]
    fig.add_trace(go.Scatter(
        x=df.index[fundos_valid],
        y=df[col].values[fundos_valid],
        mode='markers',
        name='Fundos',
        marker=dict(
            size=10,
            color=CORES['positivo'],
            symbol='triangle-up',
            line=dict(color='white', width=1)
        ),
        hovertemplate='Fundo: R$ %{y:.2f}<extra></extra>'
    ), row=1, col=1)

# Anotação preço atual
fig.add_annotation(
    x=df.index[-1],
    y=df[col].iloc[-1],
    text=f"<b>R$ {df[col].iloc[-1]:.2f}</b>",
    showarrow=True, arrowhead=0,
    ax=50, ay=0,
    font=dict(size=11, color=CORES['principal']),
    bgcolor='white',
    bordercolor=CORES['principal'],
    borderwidth=1,
    row=1, col=1
)

# ══════════════════════════════════════════════════════════════
# ROW 2 — ROLLING SHARPE
# ══════════════════════════════════════════════════════════════

sharpe_vals = df['rolling_sharpe'].dropna()

# Colorir baseado em positivo/negativo
fig.add_trace(go.Scatter(
    x=sharpe_vals.index,
    y=sharpe_vals,
    name='Rolling Sharpe',
    fill='tozeroy',
    line=dict(color=CORES['tendencia'], width=1.2),
    fillcolor='rgba(108, 92, 231, 0.2)',
    hovertemplate='Sharpe: %{y:.2f}<extra></extra>'
), row=2, col=1)

# Linhas de referência
fig.add_hline(y=0, row=2, col=1, line_color=CORES['neutro'], line_width=1.5)
fig.add_hline(y=1, row=2, col=1, line_color=CORES['positivo'], line_dash='dash', 
              line_width=1, annotation_text='Sharpe = 1', annotation_position='top left')
fig.add_hline(y=-1, row=2, col=1, line_color=CORES['negativo'], line_dash='dash', 
              line_width=1, annotation_text='Sharpe = -1', annotation_position='bottom left')

# Valor atual
sharpe_atual = sharpe_vals.iloc[-1]
cor_sharpe = CORES['positivo'] if sharpe_atual > 0 else CORES['negativo']

fig.add_annotation(
    x=sharpe_vals.index[-1],
    y=sharpe_atual,
    text=f"<b>{sharpe_atual:.2f}</b>",
    showarrow=True, arrowhead=0,
    ax=40, ay=0,
    font=dict(size=11, color=cor_sharpe),
    bgcolor='white',
    bordercolor=cor_sharpe,
    borderwidth=1,
    row=2, col=1
)

# ══════════════════════════════════════════════════════════════
# LAYOUT
# ══════════════════════════════════════════════════════════════

fig.update_layout(
    template='plotly_white',
    height=700,
    width=1400,
    margin=dict(l=60, r=60, t=80, b=40),
    hovermode='x unified',
    title=dict(
        text=f"<b>{param['ticker'].upper()}</b> — Análise Técnica e Performance",
        font=dict(size=20, color=CORES['principal']),
        x=0.5, xanchor='center'
    ),
    legend=dict(
        orientation='h', y=1.02, x=0.5, xanchor='center',
        bgcolor='rgba(255,255,255,0.9)',
        font=dict(size=10)
    ),
    font=dict(family='Segoe UI, Arial', size=10, color=CORES['principal'])
)

# Eixos
fig.update_yaxes(title_text='Preço (R$)', title_font=dict(size=10), row=1, col=1)
fig.update_yaxes(title_text='Sharpe', title_font=dict(size=10), row=2, col=1)

fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')

fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(showticklabels=True, row=2, col=1, tickformat='%b/%Y')

fig.show()

## Interpretação do Rolling Sharpe Ratio

O **Rolling Sharpe Ratio** é calculado com uma janela móvel (63 dias ≈ 3 meses) e mede o retorno ajustado ao risco ao longo do tempo.

### Níveis de Referência

| Sharpe Ratio | Interpretação |
|:------------:|---------------|
| **> 2.0** | Excelente - retorno muito alto para o risco assumido |
| **1.0 - 2.0** | Bom - retorno adequado ao risco |
| **0.5 - 1.0** | Aceitável - retorno marginal ao risco |
| **0 - 0.5** | Fraco - risco mal remunerado |
| **< 0** | Negativo - ativo está gerando perdas |

### Como Usar

- **Sharpe > 1**: Momento favorável para manter ou aumentar posição
- **Sharpe < 0 (prolongado)**: Sinal de fraqueza - considerar redução
- **Cruzamento de 0 para cima**: Possível início de tendência positiva
- **Cruzamento de 0 para baixo**: Possível início de tendência negativa

### Limitações

1. **Backward-looking**: Mede o passado, não prevê o futuro
2. **Sensível à janela**: Janelas curtas são mais ruidosas
3. **Assume normalidade**: Não captura riscos de cauda (use CVaR para complementar)
4. **Taxa livre de risco**: Cálculo assume rf=0 (simplificação)